# Equity options (Black–76 + Monte Carlo benchmark)

**Prerequisites:** **06**. **`equity_option`** with **`black76`**; compare to **`finstack.monte_carlo`** European pricer as an independent simulation check.


## Concept

- **Black–76** on forwards: discounting, dividend yield, vol surface.
- **Delta** (and other Greeks) come from the analytical pricer when registered.
- **Monte Carlo** reproduces the same risk-neutral expectation subject to simulation noise.


In [1]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.monte_carlo import EuropeanPricer

print("Imports OK (equity options).")


Imports OK (equity options).


## Instrument JSON


In [2]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

equity_option = json.dumps(
    {
        "type": "equity_option",
        "spec": {
            "id": "SPX-CALL-4500",
            "underlying_ticker": "SPX",
            "strike": 4500.0,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "notional": {"amount": 100.0, "currency": "USD"},
            "day_count": "Act365F",
            "settlement": "cash",
            "discount_curve_id": "USD-OIS",
            "spot_id": "EQUITY-SPOT",
            "vol_surface_id": "EQUITY-VOL",
            "div_yield_id": "EQUITY-DIVYIELD",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

try:
    validate_instrument_json(equity_option)
    print("equity_option JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


equity_option JSON OK


## MarketContext


In [3]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois)
state = json.loads(mc.to_json())
exp_eq = [0.25, 0.5, 1.0, 2.0]
strikes_eq = [4000.0, 4300.0, 4500.0, 4700.0, 5000.0]
state["surfaces"] = [
    {
        "id": "EQUITY-VOL",
        "expiries": exp_eq,
        "strikes": strikes_eq,
        "secondary_axis": "strike",
        "interpolation_mode": "vol",
        "vols_row_major": [0.22] * (len(exp_eq) * len(strikes_eq)),
    }
]
state.setdefault("prices", {})
state["prices"]["EQUITY-SPOT"] = {"price": {"amount": 4500.0, "currency": "USD"}}
state["prices"]["EQUITY-DIVYIELD"] = {"unitless": 0.015}
market_json = json.dumps(state)
print("surfaces:", len(json.loads(market_json)["surfaces"]))


surfaces: 1


## Pricing


In [4]:
try:
    out = price_instrument_with_metrics(
        equity_option, market_json, AS_OF_STR, model="black76", metrics=["delta"]
    )
    vr = ValuationResult.from_json(out)
    print("JSON black76:", vr)
    print("delta:", vr.get_metric("delta"))
except Exception as exc:
    print("black76:", type(exc).__name__, exc)


JSON black76: ValuationResult(id="SPX-CALL-4500", price=50511.4209, currency=USD, metrics=1)
delta: 57.43025031192922


## Metrics & MC cross-check


In [5]:
# Independent GBM Monte Carlo on a stylized underlying (not the JSON pricer MC path).
import math
from datetime import date

spot = 4500.0
strike = 4500.0
q = 0.015
vol = 0.22
# Align discounting with the same USD-OIS knot points as the market cell: DF(T) via linear interp on (t, DF).
as_of = date(2025, 1, 15)
expiry = date(2026, 6, 15)
T = (expiry - as_of).days / 365.0
ois_knots = [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)]
# Piecewise linear in tenor (years).
tenors, dfs = zip(*ois_knots)
i = 0
while i < len(tenors) - 1 and not (tenors[i] <= T <= tenors[i + 1]):
    i += 1
if i >= len(tenors) - 1:
    df_T = dfs[-1]
else:
    t0, t1 = tenors[i], tenors[i + 1]
    df0, df1 = dfs[i], dfs[i + 1]
    w = 0.0 if t1 == t0 else (T - t0) / (t1 - t0)
    df_T = df0 + w * (df1 - df0)
r = -math.log(df_T) / T
pricer = EuropeanPricer(num_paths=40_000, seed=42)
mc_res = pricer.price_call(spot=spot, strike=strike, rate=r, div_yield=q, vol=vol, expiry=T)
print("MC call (per unit notional scale): mean=", mc_res.mean.amount, "stderr=", mc_res.stderr)
print(
    "Analytical black76 from JSON above; MC uses r=-log(DF)/T from OIS knots, DF(T)≈",
    round(df_T, 6),
    "T=",
    round(T, 4),
)


MC call (per unit notional scale): mean= 507.9342898413017 stderr= 4.095763215160227
Analytical black76 from JSON above; MC uses r=-log(DF)/T from OIS knots, DF(T)≈ 0.955521 T= 1.4137


## Implied volatility override (flat σ across tenor and strike)

`pricing_overrides.implied_volatility` short-circuits the vol surface and uses a **flat σ** for pricing. It is honored uniformly across vanilla options (Black-76) and all surface-driven Monte Carlo pricers: **autocallables, cliquets, Asian, lookback, barrier, quanto, FX barrier** — anywhere the pricer previously read `get_surface(id).value_clamped(t, K)`.

This is the canonical lever for:

- **Revaluation workflows** — quote the bond/option at a specified implied vol without hand-editing a surface snapshot.
- **Vega scenarios** — shift an instrument to a flat σ level for risk reports.
- **Reproducing client-quoted prices** — use the broker's implied vol directly.

In [6]:
import copy


def price_option_at_vol(implied_vol: float | None) -> ValuationResult:
    """Reprice the SPX call with an optional flat-σ override."""
    spec = json.loads(equity_option)
    if implied_vol is not None:
        spec["spec"]["pricing_overrides"] = {"implied_volatility": implied_vol}
    out = price_instrument_with_metrics(
        json.dumps(spec), market_json, AS_OF_STR, model="black76", metrics=["delta"]
    )
    return ValuationResult.from_json(out)


# Surface σ across the grid is 0.22; override to 0.30 (fatter) and 0.15 (tighter).
for iv in (None, 0.15, 0.22, 0.30):
    vr = price_option_at_vol(iv)
    tag = "surface" if iv is None else f"flat σ = {iv:.2f}"
    print(f"{tag:<18s} price = {vr.price:>10,.4f} USD  delta = {vr.get_metric('delta'):.4f}")

surface            price = 50,511.4209 USD  delta = 57.4303
flat σ = 0.15      price = 36,220.4536 USD  delta = 57.4102
flat σ = 0.22      price = 50,511.4209 USD  delta = 57.4303
flat σ = 0.30      price = 66,793.7914 USD  delta = 58.3460


## Takeaways

- **`black76`** is the JSON model key for vanilla equity options here.
- **`finstack.monte_carlo.EuropeanPricer`** gives a simulation benchmark; align vol, div yield, rate, and horizon when comparing.
- JSON **`price_instrument_with_metrics`** does not currently register `monte_carlo_gbm` for `equity_option` in this build.
- **`implied_volatility` override** now applies uniformly across vanilla and path-dependent MC pricers as a flat σ — no more per-pricer ad-hoc handling.